# `crypto(base, start, end=None, quote="USD")`

Daily prices for the coins `crypto.prices` holds, from FMP, in USD or any currency `forex()` can price. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

In [ ]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.crypto import NAMES, CryptoBar
from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import crypto, list_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The coins
Read from the table on every call, so a coin the main repo adds or drops shows up here with no code change. The fixed top 15 have names; any other stored coin works by code, with `name` blank (the 8 dropped coins, until migration 138 deletes them).

In [ ]:
pl.DataFrame([{"coin": s.key.removesuffix("USD"), "name": NAMES.get(s.key.removesuffix("USD")), "latest_date": s.latest_date}
              for s in list_series("crypto", "prices")])

## By code or name
A name works anywhere a code does, in any case and spacing. Every bar carries the code and the name.

In [ ]:
week = today_et() - timedelta(days=7)
assert crypto("BTC", week) == crypto("bitcoin", week) == crypto("  Bitcoin ", week)
show(crypto("ETH", week)[-2:] + crypto("unus sed leo", week)[-2:])

## In USD, the default
FMP daily bars as stored (`route` = stored): open, high, low, close, `volume_usd` and VWAP. Crypto trades every day, weekends included. `quote_date` is blank because no rate is used.

In [ ]:
show(crypto("BTC", today_et() - timedelta(days=7)))

## In another currency
The USD price times `forex("USD", "EUR")` that day (`route` = crossed). Open uses the day's opening rate, close its closing rate. High, low and VWAP are blank: the coin and the rate peak at different moments. `volume_usd` stays in US dollars.

In [ ]:
show(crypto("SOL", today_et() - timedelta(days=7), quote="EUR"))

In [ ]:
start = today_et() - timedelta(days=7)
rows = [r for q in ["USD", "EUR", "JPY", "GBP", "INR", "CHF"] for r in crypto("BTC", start, quote=q)]
to_df(rows, CryptoBar).pivot(on="quote", index="date", values="close")

## Days with no rate carry the last one
Forex has no Saturday bars and FMP has short Sunday bars (the market opens Sunday evening), so most Saturdays carry Friday's close, for open and close alike, and `quote_date` names the day the rate came from. A day FMP does have a rate for uses it, such as the zero-volume Saturday bar on 2026-10-31. Here: carried days for ETH in EUR over the last year, by weekday.

In [ ]:
eth_eur = to_df(crypto("ETH", today_et() - timedelta(days=365), quote="EUR"), CryptoBar)
carried = eth_eur.filter(pl.col("quote_date") != pl.col("date"))
print(f"{carried.height} of {eth_eur.height} days carried a rate")
carried.group_by(pl.col("date").dt.strftime("%a").alias("weekday")).len().sort("len", descending=True)

## The first days look back before start
Starting on a carried day still gives that day a price: the rate is looked up as far as 14 days before `start`.

In [ ]:
first = carried["date"][-1]
show(crypto("ETH", first, first + timedelta(days=2), quote="EUR"))

## `crypto("ALL", start, end, quote)`
Every stored coin in one currency, ordered by code then date. Each symbol is read once, and the currency rate once.

In [ ]:
to_df(crypto("ALL", today_et() - timedelta(days=7)), CryptoBar).pivot(on="coin", index="date", values="close")

## Full history in one call
Bitcoin in USD from the start of FMP's history (2010). There is no row limit.

In [ ]:
btc = to_df(crypto("BTC", date(2009, 1, 1)), CryptoBar)
print(f"{btc.height:,} days, {btc['date'].min()} to {btc['date'].max()}")
(
    btc.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

## Rows are served as stored
FMP's first two DOGE bars (2017-05-31 and 06-01) are 0.0001 placeholders. They come back unchanged.

In [ ]:
show(crypto("DOGE", date(2017, 5, 31), date(2017, 6, 3)))

## Bad input
An FMP symbol instead of a code, a name that isn't on the list, `quote="ALL"`, a currency with no USD pair (a coin is not a quote currency), or a start after the end each raise a clear error.

In [ ]:
week = today_et() - timedelta(days=7)
for base, quote in [("BTCUSD", "USD"), ("litecoin", "USD"), ("BTC", "ALL"), ("BTC", "XAU"), ("BTC", "ETH")]:
    try:
        crypto(base, week, quote=quote)
    except ValueError as err:
        print(f"{base}/{quote}: {err}")
try:
    crypto("BTC", today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")